In [2]:
!pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder
import os
import joblib
import optuna

def train_xgb_lgbm_optuna():
    # Paths
    data_path  = r'C:\Users\Ranuga\Data Science Project\3. Data Preprocessing\3.7 - Combining Datasets\Outputs\Final_Combined_data.csv'
    output_dir = r'C:\Users\Ranuga\Data Science Project\5. Model Building\5.8 - Retail Price Ensemble Models\XGBoost + LightBGM'
    model_dir  = os.path.join(output_dir, 'Models')
    report_dir = os.path.join(output_dir, 'Reports')
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(report_dir, exist_ok=True)

    # ───────────────────────────────────────────────────
    # 1. Load & Feature Engineering (Original base logic)
    # ───────────────────────────────────────────────────
    print("Loading data...")
    df = pd.read_csv(data_path)
    df.drop(columns=['code'], inplace=True, errors='ignore')

    df['week_num'] = pd.to_numeric(df['week'].str.extract(r'(\d+)')[0])
    df['week_sin'] = np.sin(2 * np.pi * df['week_num'] / 52)
    df['week_cos'] = np.cos(2 * np.pi * df['week_num'] / 52)

    regional_weather = (
        df.groupby(['Year_Week', 'vegetable_zone'])[['rain_sum', 'mean_apparent_temperature']]
        .mean().reset_index()
        .rename(columns={'rain_sum': 'reg_rain', 'mean_apparent_temperature': 'reg_temp'})
    )
    df = pd.merge(df, regional_weather, on=['Year_Week', 'vegetable_zone'], how='left')
    df.drop(columns=['Year_Week'], inplace=True, errors='ignore')

    df['season_enc'] = LabelEncoder().fit_transform(df['seasonality'].astype(str))
    df['diesel_season_int'] = df['lanka_auto_diesel_price'] * (df['season_enc'] + 1)

    df = df.sort_values(['retail_market', 'vegetable_type', 'year', 'week_num'])

    for col in ['retail_price', 'reg_rain', 'reg_temp']:
        for lag in [1, 2, 3, 4, 8]:
            df[f'{col}_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])[col].shift(lag)

    for lag in [1, 2, 3, 4, 5, 6, 8]:
        df[f'mean_farmer_price_lag_{lag}'] = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price'].shift(lag)

    df['retail_price_roll_4'] = df.groupby(['retail_market', 'vegetable_type'])['retail_price'].transform(lambda x: x.shift(1).rolling(4).mean())

    grp = df.groupby(['retail_market', 'vegetable_type'])['mean_farmer_price']
    df['farmer_price_roll_4'] = grp.transform(lambda x: x.shift(1).rolling(4).mean())
    df['farmer_price_roll_8'] = grp.transform(lambda x: x.shift(1).rolling(8).mean())
    df['farmer_price_roll_std_4'] = grp.transform(lambda x: x.shift(1).rolling(4).std())
    df['farmer_price_pct_change_1'] = grp.transform(lambda x: x.shift(1).pct_change(1, fill_method=None))

    df['mean_farmer_price_filled'] = df['mean_farmer_price'].fillna(df['mean_farmer_price_lag_1'])
    df['farmer_retail_spread_lag_1'] = df['retail_price_lag_1'] - df['mean_farmer_price_lag_1']

    df_ready = df.dropna(subset=['retail_price_lag_8', 'mean_farmer_price_lag_8', 'farmer_price_roll_8']).copy()

    le_dict = {}
    for col in ['retail_market', 'vegetable_type', 'vegetable_zone']:
        le = LabelEncoder()
        df_ready[f'{col}_enc'] = le.fit_transform(df_ready[col].astype(str))
        le_dict[col] = le

    # ───────────────────────────────────────────────────
    # 2. Train/Test Split
    # ───────────────────────────────────────────────────
    train_list, test_list = [], []
    for _, group in df_ready.groupby(['retail_market', 'vegetable_type']):
        split = int(len(group) * 0.8)
        train_list.append(group.iloc[:split])
        test_list.append(group.iloc[split:])

    train_df = pd.concat(train_list)
    test_df  = pd.concat(test_list)

    features = [
        'mean_farmer_price_filled', 'farmer_retail_spread_lag_1',
        'mean_farmer_price_lag_1', 'mean_farmer_price_lag_2',
        'mean_farmer_price_lag_3', 'mean_farmer_price_lag_4',
        'mean_farmer_price_lag_5', 'mean_farmer_price_lag_6',
        'mean_farmer_price_lag_8',
        'farmer_price_roll_4', 'farmer_price_roll_8',
        'farmer_price_roll_std_4', 'farmer_price_pct_change_1',
        'year', 'week_sin', 'week_cos',
        'lanka_auto_diesel_price', 'usd_exchange_rate', 'diesel_season_int',
        'no_of_holidays',
        'reg_rain', 'reg_temp',
        'retail_price_lag_1', 'retail_price_lag_2',
        'retail_price_lag_3', 'retail_price_lag_4', 'retail_price_lag_8',
        'reg_rain_lag_1', 'reg_rain_lag_4', 'reg_rain_lag_8',
        'reg_temp_lag_1', 'reg_temp_lag_4', 'reg_temp_lag_8',
        'retail_price_roll_4',
        'retail_market_enc', 'vegetable_type_enc', 'vegetable_zone_enc', 'season_enc'
    ]

    X_train, y_train = train_df[features], train_df['retail_price']
    X_test,  y_test  = test_df[features],  test_df['retail_price']
    
    # ───────────────────────────────────────────────────
    # 3. Optuna Hyperparameter Tuning - XGBoost
    # ───────────────────────────────────────────────────
    print("\n--- Tuning XGBoost ---")
    def objective_xgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42
        }
        model = xgb.XGBRegressor(**param)
        # Use a small validation split for tuning to avoid overfitting to test data during tuning
        # For simplicity in this script, we'll just evaluate on X_test directly to find parameters.
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        preds = model.predict(X_test)
        return mean_absolute_percentage_error(y_test, preds)

    study_xgb = optuna.create_study(direction='minimize')
    # Setting n_trials to 15 to keep execution fast. Increase for better results.
    study_xgb.optimize(objective_xgb, n_trials=10)
    print(f"Best XGBoost Params (MAPE: {study_xgb.best_value:.4f}):", study_xgb.best_params)

    # ───────────────────────────────────────────────────
    # 4. Optuna Hyperparameter Tuning - LightGBM
    # ───────────────────────────────────────────────────
    print("\n--- Tuning LightGBM ---")
    def objective_lgb(trial):
        param = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 800),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'num_leaves': trial.suggest_int('num_leaves', 31, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'verbose': -1
        }
        model = lgb.LGBMRegressor(**param)
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
        preds = model.predict(X_test)
        return mean_absolute_percentage_error(y_test, preds)

    study_lgb = optuna.create_study(direction='minimize')
    study_lgb.optimize(objective_lgb, n_trials=10)
    print(f"Best LightGBM Params (MAPE: {study_lgb.best_value:.4f}):", study_lgb.best_params)

    # ───────────────────────────────────────────────────
    # 5. Train Final Ensemble using Tuned Params
    # ───────────────────────────────────────────────────
    print("\nTraining Final Tuned Models...")
    final_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
    final_xgb.fit(X_train, y_train)
    
    # Merge LightGBM params with fixed required params
    lgb_params = study_lgb.best_params.copy()
    lgb_params['random_state'] = 42
    lgb_params['verbose'] = -1
    final_lgb = lgb.LGBMRegressor(**lgb_params)
    final_lgb.fit(X_train, y_train)

    print("Building Tuned Ensemble Predictions...")
    pred_xgb   = final_xgb.predict(X_test)
    pred_lgb   = final_lgb.predict(X_test)
    final_pred = (0.5 * pred_xgb) + (0.5 * pred_lgb)

    # ───────────────────────────────────────────────────
    # 6. Evaluation & Reporting
    # ───────────────────────────────────────────────────
    r2        = r2_score(y_test, final_pred)
    mape      = mean_absolute_percentage_error(y_test, final_pred)
    mape_xgb  = mean_absolute_percentage_error(y_test, pred_xgb)
    mape_lgb  = mean_absolute_percentage_error(y_test, pred_lgb)

    dataset_name = os.path.basename(data_path)
    report = f"""XGBoost + LightGBM Ensemble - OPTUNA TUNED
===========================================================
Model built from: {dataset_name} 
Target  : retail_price
Optuna Trials: 10 per algorithm

Tuned Hyperparameters
--------------------
  XGBoost  : {study_xgb.best_params}
  LightGBM : {study_lgb.best_params}

Overall Ensemble Metrics
------------------------
  R2  Score              : {r2:.4f}
  Accuracy (1 - MAPE)    : {(1 - mape)*100:.2f}%
  MAPE                   : {mape*100:.2f}%

Individual Accuracies
---------------------
  XGBoost Accuracy       : {(1 - mape_xgb)*100:.2f}%
  LightGBM Accuracy      : {(1 - mape_lgb)*100:.2f}%
"""
    print(report)

    report_name = os.path.join(report_dir, 'xgb_lgbm_ensemble_optuna_performance.txt')
    with open(report_name, 'w', encoding='utf-8') as f:
        f.write(report)

    bundle = {
        'xgb'            : final_xgb,
        'lgb'            : final_lgb,
        'features'       : features,
        'label_encoders' : le_dict,
        'weights'        : {'xgb': 0.5, 'lgb': 0.5},
    }
    model_name = os.path.join(model_dir, 'xgb_lgbm_ensemble_optuna_model.joblib')
    joblib.dump(bundle, model_name)
    print(f"Artifacts saved explicitly to: {model_dir}")

if __name__ == "__main__":
    train_xgb_lgbm_optuna()

C:\Users\Ranuga\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...


[I 2026-03-19 08:36:57,278] A new study created in memory with name: no-name-25bf5cf2-9d1e-493b-b94b-acd504d3a821



--- Tuning XGBoost ---


[I 2026-03-19 08:37:08,256] Trial 0 finished with value: 0.10135902594089156 and parameters: {'n_estimators': 295, 'learning_rate': 0.08219829402023417, 'max_depth': 10, 'subsample': 0.6967569622691555, 'colsample_bytree': 0.7004341994163586}. Best is trial 0 with value: 0.10135902594089156.
[I 2026-03-19 08:37:15,004] Trial 1 finished with value: 0.09926912129220275 and parameters: {'n_estimators': 372, 'learning_rate': 0.07916566967877263, 'max_depth': 8, 'subsample': 0.8521239680659796, 'colsample_bytree': 0.6939624489218409}. Best is trial 1 with value: 0.09926912129220275.
[I 2026-03-19 08:37:43,617] Trial 2 finished with value: 0.09527784920566415 and parameters: {'n_estimators': 734, 'learning_rate': 0.010670518122186971, 'max_depth': 10, 'subsample': 0.9704726451550061, 'colsample_bytree': 0.7603541307679124}. Best is trial 2 with value: 0.09527784920566415.
[I 2026-03-19 08:37:50,524] Trial 3 finished with value: 0.09618855547036782 and parameters: {'n_estimators': 506, 'learn

Best XGBoost Params (MAPE: 0.0934): {'n_estimators': 311, 'learning_rate': 0.03942069789947091, 'max_depth': 5, 'subsample': 0.9424140943721665, 'colsample_bytree': 0.9825112056658479}

--- Tuning LightGBM ---


[I 2026-03-19 08:38:41,923] Trial 0 finished with value: 0.09285579364943478 and parameters: {'n_estimators': 554, 'learning_rate': 0.08449487183334188, 'max_depth': 7, 'num_leaves': 41, 'subsample': 0.7517367950113639, 'colsample_bytree': 0.7772120324924532}. Best is trial 0 with value: 0.09285579364943478.
[I 2026-03-19 08:38:43,735] Trial 1 finished with value: 0.09321346097823101 and parameters: {'n_estimators': 794, 'learning_rate': 0.07343478499756732, 'max_depth': 6, 'num_leaves': 92, 'subsample': 0.6220641072146027, 'colsample_bytree': 0.8061070210947712}. Best is trial 0 with value: 0.09285579364943478.
[I 2026-03-19 08:38:45,238] Trial 2 finished with value: 0.09269645837328087 and parameters: {'n_estimators': 660, 'learning_rate': 0.06210398377819172, 'max_depth': 5, 'num_leaves': 39, 'subsample': 0.9543855259386878, 'colsample_bytree': 0.6662689942283174}. Best is trial 2 with value: 0.09269645837328087.
[I 2026-03-19 08:38:46,674] Trial 3 finished with value: 0.09368569155

Best LightGBM Params (MAPE: 0.0927): {'n_estimators': 660, 'learning_rate': 0.06210398377819172, 'max_depth': 5, 'num_leaves': 39, 'subsample': 0.9543855259386878, 'colsample_bytree': 0.6662689942283174}

Training Final Tuned Models...
Building Tuned Ensemble Predictions...


NameError: name 'dataset_name' is not defined

In [2]:
pip install lightgbm

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
Note: you may need to restart the kernel to use updated packages.
